# 06: Linear Algebra & Matrix Operations (Exercises 76–85)

Master sliding_window_view, vector geometry projections, matrix rank, symmetric array subclasses, and patch extraction.

---


In [ ]:
import numpy as np
print(f"NumPy version: {np.__version__}")

### Exercise 76: Build a 2D array whose first row is (Z[0],Z[1],Z[2]) and subsequent rows shifted by 1
**Difficulty:** `★★★`  
**Tags:** `Stride Tricks, Sliding Window`

#### 💡 Intuition & Concept
In modern NumPy, `np.lib.stride_tricks.sliding_window_view(Z, window_shape)` creates sliding window views safely without manual pointer stride calculations.

#### ⚠️ Key Takeaway & Gotchas
The returned array is a zero-copy view into the original array buffer.


In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

Z = np.arange(10)
windows = sliding_window_view(Z, window_shape=3)
print("Array:", Z)
print("Sliding window matrix:\n", windows)

### Exercise 77: How to negate a boolean, or change the sign of a float inplace?
**Difficulty:** `★★☆`  
**Tags:** `In-place, Ufuncs`

#### 💡 Intuition & Concept
`np.logical_not(Z, out=Z)` negates booleans in-place. `np.negative(F, out=F)` negates floats in-place.

#### ⚠️ Key Takeaway & Gotchas
Using `~Z` on booleans works in Python/NumPy, but `np.logical_not` explicitly guarantees boolean output.


In [ ]:
B = np.array([True, False, True])
print("Booleans before:", B)
np.logical_not(B, out=B)
print("Booleans after: ", B)

F = np.array([1.5, -2.5, 3.0])
print("Floats before:  ", F)
np.negative(F, out=F)
print("Floats after:   ", F)

### Exercise 78: Distance from a point p to a set of 2D line segments
**Difficulty:** `★★★`  
**Tags:** `Geometry, Vector Math`

#### 💡 Intuition & Concept
Projecting point $p$ onto line segment $(P_0, P_1)$ using dot product vector projection gives the orthogonal distance.

#### ⚠️ Key Takeaway & Gotchas
Vectorizing over all line segments eliminates Python loops.


In [ ]:
def dist_point_to_lines(P0, P1, p):
    T = P1 - P0
    L = (T**2).sum(axis=1)
    u = -((P0[:, 0] - p[0]) * T[:, 0] + (P0[:, 1] - p[1]) * T[:, 1]) / L
    u = u.reshape(len(u), 1)
    d = P0 + u * T - p
    return np.hypot(d[:, 0], d[:, 1])

P0 = np.array([[0, 0], [1, 2]])
P1 = np.array([[10, 0], [1, 10]])
p  = np.array([5, 5])
distances = dist_point_to_lines(P0, P1, p)
print("Distances to lines:", distances)

### Exercise 79: Distance from a set of points P to a set of line segments
**Difficulty:** `★★★`  
**Tags:** `Geometry, Broadcasting`

#### 💡 Intuition & Concept
Expands Exercise 78 by computing distances across all combinations of points and line segments via 2D broadcasting.

#### ⚠️ Key Takeaway & Gotchas
Results in a distance matrix of shape `(num_points, num_lines)`.


In [ ]:
P0 = np.array([[0, 0], [0, 5]])
P1 = np.array([[10, 0], [10, 5]])
P  = np.array([[5, 2], [3, 4], [0, 0]])

# Point distances to lines:
dists = np.array([dist_point_to_lines(P0, P1, pt) for pt in P])
print("Shape (points x lines):", dists.shape)
print("Distances:\n", dists)

### Exercise 80: Extract a subpart with fixed shape centered on a given element (with fill padding)
**Difficulty:** `★★★`  
**Tags:** `Padding, Sub-arrays`

#### 💡 Intuition & Concept
Pad the input array generously with constant fill values, then slice around the adjusted center coordinates.

#### ⚠️ Key Takeaway & Gotchas
Padding prevents boundary condition index errors when the center is near an edge.


In [ ]:
def extract_patch(arr, center, shape, fill=0):
    r, c = center
    h, w = shape
    pad_h, pad_w = h // 2, w // 2
    padded = np.pad(arr, ((pad_h, pad_h), (pad_w, pad_w)), constant_values=fill)
    return padded[r:r + h, c:c + w]

Z = np.arange(25).reshape(5, 5)
patch = extract_patch(Z, center=(0, 0), shape=(3, 3), fill=-1)
print("Original:\n", Z)
print("Centered patch at (0,0):\n", patch)

### Exercise 81: Consider array Z = [1..14], generate rolling 4-element sub-arrays
**Difficulty:** `★★☆`  
**Tags:** `Stride Tricks, Windows`

#### 💡 Intuition & Concept
`sliding_window_view(Z, 4)` creates rolling windows of length 4.

#### ⚠️ Key Takeaway & Gotchas
Output shape is `(len(Z) - 4 + 1, 4)`.


In [ ]:
from numpy.lib.stride_tricks import sliding_window_view
Z = np.arange(1, 15)
R = sliding_window_view(Z, 4)
print("Rolling windows:\n", R)

### Exercise 82: Compute a matrix rank
**Difficulty:** `★★☆`  
**Tags:** `Linear Algebra, Matrix Rank`

#### 💡 Intuition & Concept
`np.linalg.matrix_rank` computes the number of singular values greater than numerical machine tolerance via SVD.

#### ⚠️ Key Takeaway & Gotchas
Singular matrices have rank strictly less than their minimum dimension.


In [ ]:
Z = np.eye(4)
print("Full rank (identity):", np.linalg.matrix_rank(Z))
Z[3] = Z[0]  # Linearly dependent row
print("Rank after duplicate row:", np.linalg.matrix_rank(Z))

### Exercise 83: How to find the most frequent value in an array?
**Difficulty:** `★★☆`  
**Tags:** `Statistics, Mode, Bincount`

#### 💡 Intuition & Concept
For non-negative integers, `np.bincount(Z).argmax()` returns the mode in $O(N)$ time.

#### ⚠️ Key Takeaway & Gotchas
For arbitrary floats, use `np.unique(Z, return_counts=True)`.


In [ ]:
Z = np.array([1, 2, 3, 2, 2, 4, 5, 2, 1, 4])
mode = np.bincount(Z).argmax()
print(f"Most frequent value: {mode} (appeared {np.bincount(Z)[mode]} times)")

### Exercise 84: Extract all contiguous 3x3 blocks from a 10x10 matrix
**Difficulty:** `★★☆`  
**Tags:** `Sliding Window, Image Patches`

#### 💡 Intuition & Concept
`sliding_window_view(Z, (3, 3))` creates an `(8, 8, 3, 3)` view of all overlapping 3x3 patches.

#### ⚠️ Key Takeaway & Gotchas
Commonly used in convolutional neural network feature extraction.


In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

Z = np.arange(100).reshape(10, 10)
blocks = sliding_window_view(Z, (3, 3))
print("Blocks shape:", blocks.shape)
print("First 3x3 block:\n", blocks[0, 0])

### Exercise 85: Create a 2D array subclass such that Z[i,j] == Z[j,i] (symmetric matrix)
**Difficulty:** `★★★`  
**Tags:** `OOP, Custom Array`

#### 💡 Intuition & Concept
Override `__setitem__` to assign both `(i, j)` and `(j, i)` simultaneously, guaranteeing symmetry at all times.

#### ⚠️ Key Takeaway & Gotchas
Handle tuple unpacks and slices cleanly.


In [ ]:
class SymArray(np.ndarray):
    def __setitem__(self, index, value):
        i, j = index
        super().__setitem__((i, j), value)
        super().__setitem__((j, i), value)

Z = np.zeros((4, 4)).view(SymArray)
Z[1, 2] = 42
print("Matrix:\n", Z)
print("Z[1, 2]:", Z[1, 2], "== Z[2, 1]:", Z[2, 1])
assert Z[1, 2] == Z[2, 1]